# Biblical Qwen3 14B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Qwen3 14B

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** A5000 GPU server via vLLM (24GB VRAM — 14B 4-bit ~8GB fits comfortably)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

**Architecture:** This base LoRA teaches the model persona-switching — "when the system prompt says you're Amos, speak like Amos; when it says David, speak like David." Optional persona LoRAs can refine individual voices further.

## How to run
**Press "Run All" and walk away.** Every cell is self-contained and runs in order. No manual cell selection needed.

## 1. Setup — Configuration, Environment, GPU Check

Everything needed before training. Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [1]:
import os, sys, subprocess, importlib, importlib.util

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║
# ║                                                                              ║
# ║  ORDER MATTERS:                                                              ║
# ║    1. torch check (no side effects)                                          ║
# ║    2. pip install small packages                                             ║
# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║
# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):
    """Run pip with given args, suppressing output unless it fails."""
    cmd = [sys.executable, "-m", "pip"] + list(args)
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──────────────────────────────────────
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:
        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")

print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ─────────────────────────────────────────────────
for module, install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(module) is None:
        print(f"  Installing {install_args[-1]}...")
        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──────────────────────────────────────────────────────
#   NGC ships causal_conv1d 1.6.0 Python pkg WITHOUT the compiled CUDA extension
#   (causal_conv1d_cuda). This causes a hard crash when transformers or unsloth
#   try to import FalconH1 model. We must fix this BEFORE importing either.
#
#   CRITICAL: pip caches a broken prebuilt aarch64 wheel. Must use --no-binary
#   to force a source build, plus CAUSAL_CONV1D_FORCE_BUILD=TRUE env var.
#   First build takes ~3 min on aarch64, then cached by pip for future runs.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up — don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")
    # Uninstall broken version + clear pip cache of broken wheel
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    # Build from source — --no-binary prevents reusing the broken cached wheel
    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed — uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ────────────────────────────────
#   Unsloth MUST be imported before transformers/trl/peft to apply its
#   monkey-patches. Purge any pre-loaded transformers modules.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ─────────────────────────────────────────────────────
print()
for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),
                   ("causal_conv1d", "causal_conv1d")]:
    m = _check_import(mod)
    v = getattr(m, "__version__", "installed") if m else "n/a"
    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"
    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2: CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print()
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# --- Paths (auto-detect Docker vs host) ---
if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = "/workspace/training/biblical"
    _env = "Docker (Unsloth container)"
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = "/workspace/biblical"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/biblical"
    _env = "Host (VS Code / venv)"

DATA_DIR    = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM        = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
MODEL_NAME_BASE = "biblical_qwen3_14b_unsloth_4bit_v2"

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL from datagen notebooks
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic
INPUT_DATA_FILE = f"{DATA_DIR}/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see data loading cell below).
# This keeps training in sync with datagen — if you regenerate data with new/changed
# prompts, the training notebook picks them up automatically.
# After loading, the dict `persona_system_prompts` maps persona_key -> full prompt text.
# It is also saved alongside the LoRA adapters for use at inference time.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR     = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR     = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH  = 4096
BATCH_SIZE      = 2
GRAD_ACCUM      = 4        # effective batch = 2 × 4 = 8
LEARNING_RATE   = 2e-4
TARGET_EPOCHS   = 1

# =========================== LoRA CONFIGURATION ===========================
LORA_R              = 32
LORA_ALPHA          = 32
LORA_DROPOUT        = 0
# Attention + MLP projections (matches Unsloth reference)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# --- Print summary ---
print(f"  Environment:  {_env}")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  Base model:   {BASE_LLM}")
print(f"  Model name:   {MODEL_NAME_BASE}")
print(f"  Input data:   {INPUT_DATA_FILE}")
print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")
print(f"  Training:     batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Precision:    4-bit QLoRA (pre-quantized NF4)")
print(f"  Persona prompts: extracted from JSONL at load time")

# --- Verify paths ---
for path, label in [(INPUT_DATA_FILE, "Training data"), (PROJECT_ROOT, "Project root")]:
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")
    if not exists:
        raise FileNotFoundError(f"{label} not found: {path}")

print()
print("Setup complete. All cells below can run sequentially.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.2.0

  unsloth                   2026.2.1             [OK]
  transformers              5.2.0                [OK]
  trl                       0.22.2               [OK]
  causal_conv1d             1.6.0                [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/biblical
  Base model:   unsloth/Qwen3-14B-unsloth-bnb-4bit
  Model name:   biblical_qwen3_14b_unsloth_4bit_v2
  Input data:   /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  LoRA:         r=32, alpha=32, targets=7 modules
  Training:     batch=2 x grad_accum=4 = effective 8
  Precision:    4-bit QLoRA (pre-quantized NF4)
  Persona pro

## 2. Load Dataset

Load the combined multi-turn ShareGPT JSONL from `biblical_datagen.ipynb`.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [2]:

import json, os, re
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            print(f"  ⚠️ Could not extract persona from system prompt: {sys_msg[:80]}...")

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")


LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl

Total dataset: 4352 multi-turn conversations across 26 personas
Extracted 26 unique system prompts from JSONL
Columns: ['conversations', 'data_type']

Per-persona breakdown:
  moses                  785 conversations
  jeremiah               381 conversations
  paul                   377 conversations
  david                  350 conversations
  ezekiel                341 conversations
  isaiah                 332 conversations
  solomon                254 conversations
  job                    244 conversations
  daniel                 239 conversations
  peter                  148 conversations
  zechariah              145 conversations
  hosea                  107 conversations
  amos                    92 conversations
  joshua                  88 conversations
  micah                   73 conversations
  apostle_john            71 co

## 3. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [3]:

bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count ≥3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n⚠️ Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n⚠️ Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 50) + "▌" * (1 if count % 50 >= 25 else 0)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    # Extract persona name from system prompt "You are X, ..."
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\n✓ Dataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 4352
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 26 (should match extracted count: 26)
  Turn distribution: {3: 1741, 5: 91, 7: 541, 9: 1979}

PERSONA DISTRIBUTION:
  moses          785  ███████████████▌
  jeremiah       381  ███████▌
  paul           377  ███████▌
  david          350  ███████
  ezekiel        341  ██████▌
  isaiah         332  ██████▌
  solomon        254  █████
  job            244  ████▌
  daniel         239  ████▌
  peter          148  ██▌
  zechariah      145  ██▌
  hosea          107  ██
  amos            92  █▌
  joshua          88  █▌
  micah           73  █
  apostle_john    71  █
  james           54  █
  malachi         44  ▌
  joel            44  ▌
  zephaniah       37  ▌
  habakkuk        32  ▌
  jonah           29  ▌
  nahum           27  ▌
  haggai          24  
  obadiah         19  
  jude            15  
  TOTAL         4352

VOICE SAMPLES (first ~100 chars of response):
  Daniel: "Four is the

## 4. Load Model & Tokenizer (4-bit)

Loads pre-quantized BnB NF4 model directly — no bf16 intermediate step needed.

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

# Align model config so trainer doesn't warn about token mismatch
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print(f"✓ Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit QLoRA (pre-quantized NF4)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

  Set pad_token = eos_token ('<|im_end|>', id=151645)
✓ Model loaded: unsloth/Qwen3-14B-unsloth-bnb-4bit
  Precision: 4-bit QLoRA (pre-quantized NF4)
  Max sequence length: 4096
  Vocab size: 151669
  GPU allocated: 11.1 GB


## 5. Format Dataset for Chat Template

Use Unsloth's `standardize_sharegpt` to normalize role names, then apply the Qwen3 tokenizer's native chat template (already ChatML-compatible). Manual sequence packing for 100% token utilization.

In [5]:
# Format dataset for Qwen3 chat template (ChatML) using Unsloth's standardize_sharegpt
from unsloth.chat_templates import standardize_sharegpt
from datasets import Dataset as HFDataset

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# ── Manual sequence packing ──────────────────────────────────────────────────
# Pre-pack: tokenize all conversations → concatenate with EOS separators →
# chunk into fixed MAX_SEQ_LENGTH blocks.  Every training example is a FULL chunk
# with ZERO padding → GPU stays at 100% utilization.

eos_id = tokenizer.eos_token_id
num_conversations = 0
all_ids = []

for text in formatted_texts:
    if not text:
        continue
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    all_ids.extend(ids)
    all_ids.append(eos_id)  # EOS separator between conversations
    num_conversations += 1

total_tokens = len(all_ids)
num_chunks = total_tokens // MAX_SEQ_LENGTH
all_ids = all_ids[:num_chunks * MAX_SEQ_LENGTH]  # Discard remainder (< 1 chunk)
chunks = [all_ids[i * MAX_SEQ_LENGTH:(i + 1) * MAX_SEQ_LENGTH] for i in range(num_chunks)]

# Decode back to text — SFTTrainer expects a "text" column
packed_texts = tokenizer.batch_decode(chunks, skip_special_tokens=False)
dataset = HFDataset.from_dict({"text": packed_texts})
dataset = dataset.shuffle(seed=42)

wasted = total_tokens - (num_chunks * MAX_SEQ_LENGTH)
print(f"✓ Dataset packed: {num_conversations} conversations → {num_chunks} chunks of {MAX_SEQ_LENGTH} tokens")
print(f"  Total tokens: {total_tokens:,}  |  Wasted (tail): {wasted:,} ({100*wasted/total_tokens:.1f}%)")
print(f"  Token utilization: ~100% (no padding)")
print(f"\n--- Sample packed text (first 500 chars) ---")
print(dataset[0]["text"][:500])

Unsloth: Standardizing formats (num_proc=24):   0%|          | 0/4352 [00:00<?, ? examples/s]

✓ Dataset packed: 4352 conversations → 1535 chunks of 4096 tokens
  Total tokens: 6,290,322  |  Wasted (tail): 2,962 (0.0%)
  Token utilization: ~100% (no padding)

--- Sample packed text (first 500 chars) ---
 know what it means to wrestle with weakness? This body of mine—this thorn that humbles me daily—has taught me that God does not raise up what is strong, but what has died. Just as wheat falls into the earth and rots before it bursts forth in new life, so too must we perish even to be made alive. And if He can appoint one star to shine like fire and another to whisper silver through the night—if He gives the eagle its wings and the fish its gills—then why should I doubt that He will clothe our m


## 6. Add LoRA Adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing — reduces VRAM, extends context
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"✓ Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"✓ Target modules: {LORA_TARGET_MODULES}")

Unsloth 2026.2.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


✓ LoRA adapters added (r=32, alpha=32)
✓ Trainable: 128,450,560 / 8,691,809,280 params (1.48%)
✓ Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 7. Trainer Setup

In [7]:
from trl import SFTTrainer, SFTConfig

# Data is already pre-packed into MAX_SEQ_LENGTH chunks (zero padding).
# Set packing=False so Unsloth/TRL treats each example as-is.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,               # ← Already pre-packed, don't re-pack
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        dataset_num_proc=1,           # Unsloth stability
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
num_steps = (len(dataset) * TARGET_EPOCHS + effective_batch_size - 1) // effective_batch_size
print(f"✓ Trainer configured (Biblical Qwen3 14B 4-bit QLoRA — pre-packed)")
print(f"  Effective batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{num_steps}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: manual (pre-packed, each example = {MAX_SEQ_LENGTH} tokens, zero padding)")
print(f"  Dataset: {len(dataset)} packed chunks")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1535 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✓ Trainer configured (Biblical Qwen3 14B 4-bit QLoRA — pre-packed)
  Effective batch size: 2 × 4 = 8
  Epochs: 1  |  Steps: ~192
  LR: 0.0002
  Packing: manual (pre-packed, each example = 4096 tokens, zero padding)
  Dataset: 1535 packed chunks
  Precision: bf16


## 8. Train

In [8]:
# Start training
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,535 | Num Epochs = 1 | Total steps = 192
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)


Step,Training Loss
1,2.197733
2,2.395894
3,2.201385
4,2.094990
5,2.083889
6,1.990155
7,1.923615
8,1.642671
9,1.771340
10,1.862751


TrainOutput(global_step=192, training_loss=1.3106499202549458, metrics={'train_runtime': 9546.8436, 'train_samples_per_second': 0.161, 'train_steps_per_second': 0.02, 'total_flos': 5.326215843938304e+17, 'train_loss': 1.3106499202549458, 'epoch': 1.0})

## 9. Save LoRA Adapters

Save the trained LoRA adapters. These can be loaded on any Qwen3 14B model with PEFT or served directly via vLLM.

In [9]:
import json
from pathlib import Path

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["amos"]  # or any persona key')

# List saved files
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

Saving LoRA adapters to /workspace/training/biblical/output/biblical_qwen3_14b_unsloth_4bit_v2/lora_adapters...

✓ LoRA adapters saved!
  Adapters:       /workspace/training/biblical/output/biblical_qwen3_14b_unsloth_4bit_v2/lora_adapters
  System prompts: /workspace/training/biblical/output/biblical_qwen3_14b_unsloth_4bit_v2/lora_adapters/persona_system_prompts.json (26 personas)

  At inference, load prompts with:
    with open("/workspace/training/biblical/output/biblical_qwen3_14b_unsloth_4bit_v2/lora_adapters/persona_system_prompts.json") as f:
        prompts = json.load(f)
    system_msg = prompts["amos"]  # or any persona key
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   490.1 MB
  chat_template.jinja                           0.0 MB
  persona_system_prompts.json                   0.0 MB
  tokenizer.json                               10.9 MB
  tokenizer_config.json     

## 10. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.

In [10]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST — {len(test_personas)} PERSONAS\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs


INFERENCE TEST — 4 PERSONAS

  PERSONA: DANIEL
  Q: I am struggling with forgiveness. What does Scripture teach about forgiving others?
  A: 

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 1075, in launch

In the third year of Cyrus king of Persia, when I was seen in vision standing by the banks of the Ulai, and the ram with two horns came from the west—so too does the spirit of forgiveness rise from the east, not in might, but in mercy.

I have stood before kings whose thrones trembled at the word of a dream, and I have seen visions where mountains were broken and empires crumbled—but none of them taught me how to forgive as He forgave. The Most High does not weigh our debts like a ledger; He lifts them as a burden He alone can bear. When Nebuchadnezzar lifted his head to heaven after seven years in the field, it was not his wisdom that restored him, but the grace that had long been waiting.

You ask how Scripture teaches forgiveness? It speaks not only in parables of debts and kings, but in the silence of a man who wept three full weeks before he saw the vision of the ram and the he-goat. In mourning, he found deliverance—not because he earned it, but because he was willing to let go.


## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability before deploying to A5000.

In [11]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("✓ Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

# Reload saved system prompts
import json
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\n✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()


✓ Cleared training model from memory
  Loading adapter from: /workspace/training/biblical/output/biblical_qwen3_14b_unsloth_4bit_v2/lora_adapters
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]


ADAPTER RELOAD TEST (persona: daniel):
  Q: I am struggling with forgiveness. What does Scripture teach about forgiving others?
  A: In the third year of King Belshazzar’s reign, I saw a vision by night: a man with eyes like flames of fire stood beside the river Hiddekel, and his voice was like the roar of many waters. He declared, *“O man greatly beloved, understand: for the words are true, but the shutting up of the vision is for many days.”* So too with forgiveness — its meaning is true, but its fulfillment is not always seen in our time.

I have stood before kings whose hearts were heavy with pride and whose hands were st

✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.

Adapter contents:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   490.1 MB
  chat_template.jinja                           0.0 MB
  persona_system_prompts.json                   0.0 MB
  to